In [ ]:
import pandas as pd
from pathlib import Path

# Paths de tus dos archivos (ajústalos si cambian)
FILE1 = Path("S&P 500 Historical Data_2000-2019.csv")
FILE2 = Path("S&P 500 Historical Data2011-2025.csv")


def load_sp500_csv(path: Path) -> pd.DataFrame:
    """Lee y limpia un CSV del S&P 500 con formato tipo Investing.com"""
    
    df = pd.read_csv(path)

    # Normalizar nombres por si difieren entre archivos
    rename_map = {
        "Date": "Date",
        "Price": "Close",
        "Open": "Open",
        "High": "High",
        "Low": "Low",
    }

    df = df.rename(columns=rename_map)

    # Convertir fecha
    df["Date"] = pd.to_datetime(df["Date"])

    # Limpiar números con comas
    for col in ["Close", "Open", "High", "Low"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "")
            .astype(float)
        )

    # Mantener solo columnas necesarias
    df = df[["Date", "Open", "High", "Low", "Close"]]

    return df


def merge_sp500(file1: Path, file2: Path) -> pd.DataFrame:
    """Carga, limpia y combina dos archivos históricos del S&P 500."""
    
    df1 = load_sp500_csv(file1)
    df2 = load_sp500_csv(file2)

    # Unir todo
    df = pd.concat([df1, df2], ignore_index=True)

    # Eliminar duplicados por fecha
    df = df.drop_duplicates(subset="Date")

    # Ordenar cronológicamente
    df = df.sort_values("Date").reset_index(drop=True)

    # Crear target: 1 si la próxima semana baja, 0 si sube
    df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

    # remover la última fila que no tiene target
    df = df.iloc[:-1]

    return df


# ==========
# Ejecutar
# ==========
df = merge_sp500(FILE1, FILE2)
pretty_json(df.head(5))
pretty_json(df.tail(5))

        Date    Open    High     Low   Close  Target
0 2000-01-03  1469.2  1478.0  1438.4  1455.2       0
1 2000-01-04  1455.2  1455.2  1397.4  1399.4       1
2 2000-01-05  1399.4  1413.3  1377.7  1402.1       1
3 2000-01-06  1402.1  1411.9  1392.0  1403.5       1
4 2000-01-07  1403.5  1441.5  1400.5  1441.5       1
           Date     Open     High      Low    Close  Target
6505 2025-11-12  6867.77  6869.91  6829.62  6850.92       0
6506 2025-11-13  6826.47  6828.05  6724.72  6737.49       0
6507 2025-11-14  6672.14  6774.31  6646.87  6734.11       0
6508 2025-11-17  6713.61  6754.50  6638.90  6672.41       0
6509 2025-11-18  6641.19  6666.63  6574.32  6617.37       1


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd
import pandas as pd
from typing import Tuple, Dict, Any

def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    close = df["Close"]

    # MACD (12, 26, 9)
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

    # RSI 14
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()
    rs = avg_gain / avg_loss
    df["RSI_14"] = 100 - (100 / (1 + rs))

    return df



def make_weekly_dataset(df_daily_ta: pd.DataFrame) -> pd.DataFrame:
    # Ensure date is index
    df = df_daily_ta.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date")

    # Resample to weekly (Friday close)
    weekly = df.resample("W-FRI").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "MACD": "last",
        "MACD_signal": "last",
        "RSI_14": "last",
        # "Volume": "sum",   # if available
    })

    # Weekly % return
    weekly["Weekly_return"] = weekly["Close"].pct_change()

    # Target: did next week go UP?
    weekly["Target"] = (weekly["Close"].shift(-1) > weekly["Close"]).astype(int)

    # Pandemic indicator
    weekly["Pandemic"] = weekly.index.to_series().between(
        "2019-12-01", "2021-12-31"
    ).astype(int)

    # -----------------------------
    #  ⭐ ADD WEEK IDENTIFIERS ⭐
    # -----------------------------

    # Year for each week
    weekly["Year"] = weekly.index.year

    # ISO week number (1–52)
    weekly["Week_Number"] = weekly.index.isocalendar().week.astype(int)

    # Unique identifier (Ex: "2023-47")
    weekly["Week_ID"] = (
        weekly["Year"].astype(str) + "-" + weekly["Week_Number"].astype(str).str.zfill(2)
    )

    # Remove rows with NaN
    weekly = weekly.dropna()

    return weekly



import yfinance as yf

# Fetch stock data from Yahoo Finance

def get_stock_price(ticker,history=5):
    # time.sleep(4) #To avoid rate limit error
    ticker = ticker.upper().strip()
    stock = yf.Ticker(ticker)
    df = stock.history(period="1y")
    df=df[["Close","Volume"]]
    df.index=[str(x).split()[0] for x in list(df.index)]
    df.index.rename("Date",inplace=True)
    df=df[-history:]
    # pretty_json(df.columns)
    
    return df.to_string()

# Fetch financial statements from Yahoo Finance
def get_financial_statements(ticker):
    # time.sleep(4) #To avoid rate limit error
    ticker = ticker.upper().strip()    
    company = yf.Ticker(ticker)
    balance_sheet = company.balance_sheet
    if balance_sheet.shape[1]>=3:
        balance_sheet=balance_sheet.iloc[:,:3]    # Remove 4th years data
    balance_sheet=balance_sheet.dropna(how="any")
    balance_sheet = balance_sheet.to_string()
    
    # cash_flow = company.cash_flow.to_string()
    # display(balance_sheet)
    # display(cash_flow)
    return balance_sheet



def analyze_week(
    df_weekly: pd.DataFrame,
    analysis_week_end: str | pd.Timestamp | None = None,
    years_back: int = 3,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    """
    Weekly seasonal analysis using:

    - Week number based on the requested date (calendar week).
    - Historical window defined in WEEKS, not days.
    - If there is a gap between requested_date and last_available_date,
      the usable data window (in weeks) is reduced so that:
        gap_weeks + data_weeks ≈ years_back * 52
    """

    if df_weekly.empty:
        raise ValueError("df_weekly is empty – no data available for analysis.")

    df = df_weekly.sort_index().copy()

    # 1) Resolve requested date & calendar week number
    today = pd.Timestamp.today().normalize()

    if analysis_week_end is None:
        requested_date = today
    else:
        requested_date = pd.to_datetime(analysis_week_end).normalize()

    requested_week_num = int(requested_date.isocalendar().week)

    # 2) Data coverage and weekly gap
    first_date = df.index.min()
    last_date  = df.index.max()

    # Convert to weekly periods with same anchor "W-FRI"
    weekly_periods   = df.index.to_period("W-FRI")
    last_period      = weekly_periods.max()
    requested_period = requested_date.to_period("W-FRI")

    # Gap in weeks between requested week and last week in the data (INT)
    gap_weeks = max(0, requested_period.ordinal - last_period.ordinal)

    # 3) Build effective historical window in weeks
    total_weeks = int(round(years_back * 52))        # approx weeks in years_back
    data_weeks  = max(0, total_weeks - gap_weeks)    # usable weeks of real data

    if data_weeks == 0:
        raise ValueError(
            f"Requested horizon of {years_back} years cannot be satisfied: "
            f"gap of {gap_weeks} weeks is too large vs available data."
        )

    # Last period is the last available week in the dataset
    window_end_period   = last_period
    # We want data_weeks weeks including window_end_period
    window_start_period = window_end_period - (data_weeks - 1)

    # Convert back to timestamps (aligned to Friday)
    window_start = window_start_period.to_timestamp("W-FRI")
    window_end   = window_end_period.to_timestamp("W-FRI")

    # Clip to first_date in case of extreme shifts
    if window_start < first_date:
        window_start = first_date

    hist_window = df.loc[window_start:window_end]

    if hist_window.empty:
        raise ValueError(
            f"No historical data in computed weekly window "
            f"[{window_start.date()} : {window_end.date()}]."
        )

    # 4) Filter by requested calendar week number (seasonality)
    hist = hist_window[hist_window["Week_Number"] == requested_week_num].copy()
    if hist.empty:
        # Fallback: use all weeks in that window
        hist = hist_window.copy()

    # 5) Stats over the selected weeks
    mean_ret = hist["Weekly_return"].mean()
    std_ret  = hist["Weekly_return"].std()
    min_ret  = hist["Weekly_return"].min()
    max_ret  = hist["Weekly_return"].max()

    prob_up = hist["Target"].mean()  # 1 = next week up

    # Optional: last 1 year horizon, also in weeks
    total_weeks_1y = 52
    data_weeks_1y  = max(0, total_weeks_1y - gap_weeks)

    if data_weeks_1y > 0:
        window_start_1y_period = window_end_period - (data_weeks_1y - 1)
        window_start_1y = window_start_1y_period.to_timestamp("W-FRI")
        if window_start_1y < first_date:
            window_start_1y = first_date

        last_year_window = df.loc[window_start_1y:window_end]
        hist_last_year = last_year_window[last_year_window["Week_Number"] == requested_week_num]
        if hist_last_year.empty:
            hist_last_year = last_year_window

        prob_up_last_year = float(hist_last_year["Target"].mean()) if not hist_last_year.empty else float("nan")
    else:
        prob_up_last_year = float("nan")

    # 🔹 New: "analysis week end" (calendar week we are conceptually analyzing)
    analysis_week_period = requested_date.to_period("W-FRI")
    analysis_week_end_ts = analysis_week_period.to_timestamp("W-FRI")

    report: Dict[str, Any] = {
        # Conceptual week we are analyzing / forecasting
        "analysis_week_end": analysis_week_end_ts,
        "week_index": analysis_week_end_ts,   # for backward compatibility

        # Calendar info
        "requested_date": requested_date,
        "requested_week_number": requested_week_num,

        # Data coverage
        "data_first_available": first_date,
        "data_last_available": last_date,
        "gap_weeks": int(gap_weeks),
        "years_back_param": years_back,
        "window_start": window_start,
        "window_end": window_end,

        # Historical stats
        "historical_stats": {
            "num_samples": int(len(hist)),
            "mean_weekly_return": float(mean_ret),
            "std_weekly_return": float(std_ret),
            "min_weekly_return": float(min_ret),
            "max_weekly_return": float(max_ret),
        },

        # Forecast-style probabilities
        "forecast": {
            "probability_next_week_up_all_history": float(prob_up),
            "probability_next_week_up_last_year": float(prob_up_last_year),
        },
    }

    return report, hist

In [ ]:
df_daily_ta = add_technical_indicators(df.copy())
df_weekly = make_weekly_dataset(df_daily_ta)


display(df_weekly)


display(df_weekly.columns)
display(df_daily_ta.columns)
display(df_weekly.columns)



FEATURES = ["Weekly_return", "MACD", "MACD_signal", "RSI_14", "Pandemic"]

X = df_weekly[FEATURES]
y = df_weekly["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, shuffle=False, test_size=0.2
)

clf = LogisticRegression()
clf.fit(X_train, y_train)

# display(classification_report(y_test, clf.predict(X_test)))


def predict_next_week_direction(df_weekly, clf, features=FEATURES):
    last_row = df_weekly.iloc[[-1]]
    prob_up = clf.predict_proba(last_row[features])[0, 1]
    direction = "UP" if prob_up >= 0.5 else "DOWN"
    return direction, float(prob_up), last_row.index[0].date()

direction_hist, prob_up_hist, last_week_end = predict_next_week_direction(df_weekly, clf)
# display(f"Histórico → semana posterior a {last_week_end}: {direction_hist}, prob subir = {prob_up_hist:.3f}")


               Open     High      Low    Close       MACD  MACD_signal  \
Date                                                                     
2000-01-28  1441.40  1454.20  1356.10  1360.20 -14.440792    -7.196541   
2000-02-04  1360.20  1436.00  1350.00  1424.40  -9.460417   -10.410849   
2000-02-11  1424.40  1444.40  1379.20  1387.10  -7.563147    -7.737411   
2000-02-18  1387.10  1407.80  1345.30  1346.10 -14.323163   -10.002726   
2000-02-25  1346.10  1370.10  1329.10  1333.40 -20.898756   -15.206626   
...             ...      ...      ...      ...        ...          ...   
2025-10-24  6690.05  6807.11  6655.69  6791.69  38.354764    35.967525   
2025-10-31  6845.46  6920.34  6814.26  6840.20  62.295360    52.324275   
2025-11-07  6882.32  6882.32  6631.44  6728.80  33.391319    47.655962   
2025-11-14  6785.36  6869.91  6646.87  6734.11  24.032544    37.159065   
2025-11-21  6713.61  6754.50  6574.32  6617.37   0.393479    25.992506   

               RSI_14  Weekly_return 

In [ ]:
#code for get lll sentimen response
## Source - https://stackoverflow.com/a/77998581
# Posted by Nikita Malviya, modified by community. See post 'Timeline' for change history
# Retrieved 2025-11-19, License - CC BY-SA 4.0
#pip install langchain-community langchain-core
%load_ext dotenv
%dotenv
NEWS_SOURCES = [
    "https://www.reuters.com/markets/us/",
    "https://www.cnbc.com/us-markets/",
    "https://www.bloomberg.com/markets",
    "https://finviz.com/news.ashx",
    "https://coinmarketcap.com/community/es/",
    "https://stockstory.org/",
    "https://www.tipranks.com/news"
]

import os
import time
from bs4 import BeautifulSoup
import re
import requests
import warnings

import os

from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain_core.tools import Tool  # for Tool()
from langchain_core.messages import SystemMessage, HumanMessage
import langchain, langchain_core
display("langchain:", langchain.__version__, langchain.__file__)
display("langchain_core:", langchain_core.__version__)
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import Tool
langchain.verbose = False
langchain.debug = False

import json

warnings.filterwarnings("ignore")





llm = ChatOpenAI(
    temperature=1,
    model_name="gpt-5-nano-2025-08-07",
)


# Script to scrap top5 googgle news for given company name

def google_query(search_term):
    if "news" not in search_term:
        search_term=search_term+" stock news"
    url=f"https://www.google.com/search?q={search_term}&cr=countryIN"
    url=re.sub(r"\s","+",url)
    return url

def get_recent_stock_news(company_name):
    # time.sleep(4) #To avoid rate limit error
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36'}

    g_query=google_query(company_name)
    res=requests.get(g_query,headers=headers).text
    soup=BeautifulSoup(res,"html.parser")
    news=[]
    for n in soup.find_all("div","n0jPhd ynAwRc tNxQIb nDgy9d"):
        news.append(n.text)
    for n in soup.find_all("div","IJl0Z"):
        news.append(n.text)


    if len(news)>6:
        news=news[:4]
    else:
        news=news
    news_string=""
    for i,n in enumerate(news):
        news_string+=f"{i}. {n}\n"
    top5_news="Recent News:\n\n"+news_string
    
    return top5_news

def fetch_headlines(sources=NEWS_SOURCES, max_per_site=10):
    headlines = []
    for url in sources:
        try:
            res = requests.get(
                url,
                timeout=10,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            soup = BeautifulSoup(res.text, "html.parser")
            # muy genérico: h1, h2, h3
            count = 0
            for tag in soup.find_all(["h1", "h2", "h3"]):
                text = tag.get_text(strip=True)
                if len(text) > 25:
                    headlines.append(f"{url} :: {text}")
                    count += 1
                    if count >= max_per_site:
                        break
        except Exception as e:
            display(f"Error leyendo {url}: {e}")
    return headlines

search=DuckDuckGoSearchRun()
# Making tool list

tools=[
    Tool(
        name="get stock data",
        func=get_stock_price,
        description="Use when you are asked to evaluate or analyze a stock. This will output historic share price data. You should input the the stock ticker to it "
    ),
    Tool(
        name="DuckDuckGo Search",
        func=search.run,
        description="Use only when you need to get NSE/BSE stock ticker from internet, you can also get recent stock related news. Dont use it for any other analysis or task"
    ),
    Tool(
        name="get recent news",
        func=get_recent_stock_news,
        description="Use this to fetch recent news about stocks"
    ),

    Tool(
        name="get financial statements",
        func=get_financial_statements,
        description="Use this to get financial statement of the company. With the help of this data companys historic performance can be evaluaated. You should input stock ticker to it"
    ) 


]

''' function=[
        {
        "name": "get_company_Stock_ticker",
        "description": "This will get the indian NSE/BSE stock ticker of the company",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker_symbol": {
                    "type": "string",
                    "description": "This is the stock symbol of the company.",
                },

                "company_name": {
                    "type": "string",
                    "description": "This is the name of the company given in query",
                }
            },
            "required": ["company_name","ticker_symbol"],
        },
    }
] '''

def get_stock_ticker(query):
    function_def = {
        "name": "get_company_Stock_ticker",
        "description": "Extract the company name and stock ticker.",
        "parameters": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string",
                    "description": "Company name from the query.",
                },
                "ticker_symbol": {
                    "type": "string",
                    "description": "Extracted stock ticker symbol.",
                },
            },
            "required": ["company_name", "ticker_symbol"],
        },
    }

    response = llm.invoke(
        
        input=query,
        functions=[function_def],
        function_call={"name": "get_company_Stock_ticker"}
    )

    # Extract function call arguments
    func_args = response.additional_kwargs["function_call"]["arguments"]
    parsed = json.loads(func_args)

    return parsed["company_name"], parsed["ticker_symbol"]



def Anazlyze_stock(query):
    #agent.run(query) Outputs Company name, Ticker
    Company_name,ticker=get_stock_ticker(query)
    display({"Query":query,"Company_name":Company_name,"Ticker":ticker})
    stock_data=get_stock_price(ticker,history=10)
    stock_financials=get_financial_statements(ticker)
    stock_news=get_recent_stock_news(Company_name)

    # available_information=f"Stock Price: {stock_data}\n\nStock Financials: {stock_financials}\n\nStock News: {stock_news}"
    available_information=f"Stock Financials: {stock_financials}\n\nStock News: {stock_news}"

    display("\n\nAnalyzing.....\n")
    analysis=llm(f"Give detail stock analysis, Use the available data and provide investment recommendation. \
             The user is fully aware about the investment risk, dont include any kind of warning like 'It is recommended to conduct further research and analysis or consult with a financial advisor before making an investment decision' in the answer \
             User question: {query} \
             You have the following information available about {Company_name}. Write (5-8) pointwise investment analysis to answer user query, At the end conclude with proper explaination.Try to Give positives and negatives  : \
              {available_information} "
             )
    display(analysis)

    return analysis

import json

def summarize_market_sentiment(headlines, week_end_date):
    """
    Devuelve un dict:
      { "label": "bullish|bearish|neutral", "score": float, "explanation": str }
    """
    if not headlines:
        return {
            "label": "neutral",
            "score": 0.0,
            "explanation": "No headlines fetched."
        }

    text_block = "\n".join(f"- {h}" for h in headlines[:20])

    prompt = f"""
You are a professional equity analyst.

I will give you recent news headlines related to the US stock market.
Your task:

1. Classify overall sentiment for the S&P 500 for the week ending on {week_end_date} as one of:
   - bullish
   - bearish
   - neutral
2. Give a confidence score between 0 and 1.
3. Briefly explain the main reasons.

Headlines:
{text_block}

Return JSON ONLY in this format:
{{
  "label": "bullish",
  "score": 0.78,
  "explanation": "..."
}}
"""

    resp = llm.invoke(prompt)
    content = resp.content.strip()

    try:
        data = json.loads(content)
    except Exception:
        data = {
            "label": "neutral",
            "score": 0.0,
            "explanation": content[:400],
        }
    return data



The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
langchain: 0.2.10 d:\Python310\lib\site-packages\langchain\__init__.py
langchain_core: 0.2.43


In [ ]:
news_headlines = fetch_headlines()
len(news_headlines), news_headlines[:5]

display("-----------------------------------")
display(news_headlines)
display("-----------------------------------")
display(get_recent_stock_news("spx"))
display("-----------------------------------")
search("Stock news USA")
display("-----------------------------------")

from datetime import date

today = date.today()  # p.ej. 2025-11-21
# sentiment = summarize_market_sentiment(news_headlines, today)
# display(sentiment)

-----------------------------------
['https://www.bloomberg.com/markets :: Netflix, WB Discovery, Microsoft', 'https://www.bloomberg.com/markets :: Netflix, WB Discovery, Microsoft', 'https://finviz.com/news.ashx :: Upgrade your FINVIZ experience', 'https://stockstory.org/ :: Why SentinelOne (S) Stock Is Falling Today', 'https://stockstory.org/ :: FuelCell Energy (FCEL) Stock Trades Up, Here Is Why', 'https://stockstory.org/ :: Why Parsons (PSN) Shares Are Trading Lower Today']
-----------------------------------
Recent News:


-----------------------------------


d:\Python310\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
d:\Python310\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


-----------------------------------


In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_fed_page(url: str) -> str:
    """
    Downloads a Federal Reserve webpage and returns clean text.
    Works for most static HTML pages on federalreserve.gov.
    """
    headers = {"User-Agent": "Mozilla/5.0"}

    r = requests.get(url, headers=headers, timeout=10)
    r.raise_for_status()  # raise error if bad response

    soup = BeautifulSoup(r.text, "lxml")

    # Remove scripts, styles, menus
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
        tag.decompose()

    text = soup.get_text("\n", strip=True)

    return text


import json

def summarize_macro_fomc(raw_text: str, as_of_date, llm=None):
    """
    Summarizes macro + FOMC information with a simple, rule-based macro index.

    Output JSON (example):
    {
      "macro_sentiment": "hawkish | dovish | neutral",
      "macro_score": 0.30,              # from -1 (bad) to +1 (good)
      "regime": "good | ok | bad",
      "drivers": {
        "cpi_mom": 0.3,
        "cpi_yoy": 2.9,
        "unemployment": 4.4,
        "gdp_growth": 3.8,
        "policy_rate_midpoint": 3.88
      },
      "explanation": "...",
      "as_of": "YYYY-MM-DD"
    }
    """

    if llm is None:
        from langchain_openai import ChatOpenAI
        # Lower temperature → more stable classifications
        llm = ChatOpenAI(model_name="gpt-5-nano-2025-08-07", temperature=0.2)

    prompt = f"""
You are a macroeconomic analyst specializing in Federal Reserve and US macro data.

You will receive a big chunk of text that may contain:
- Policy rate (Fed funds target range)
- CPI (month-over-month and year-over-year)
- PCE inflation
- Unemployment rate
- GDP growth
- Fed communications / press releases

TEXT (as of {as_of_date}):
--------------------
{raw_text[:6000]}
--------------------

1) First, try to extract these numeric values IF POSSIBLE (if not found, use null):
   - cpi_mom: CPI month-over-month percent change (e.g. 0.3 for 0.3%)
   - cpi_yoy: CPI year-over-year percent change
   - unemployment: unemployment rate (percent)
   - gdp_growth: real GDP growth annualized (percent)
   - policy_rate_midpoint: midpoint of Fed funds target range (e.g. 3.875 if 3.75–4.00)

2) Then build a simple macro_score between -1 and +1 using this rule-of-thumb:

   Start: macro_score = 0

   Inflation (use cpi_yoy if available, otherwise approximate from text):
   - If cpi_yoy > 3.5 → macro_score -= 0.4   (inflation too high = worse)
   - Else if 2.0 <= cpi_yoy <= 3.5 → macro_score -= 0.1  (a bit above target)
   - Else if 1.0 <= cpi_yoy < 2.0 → macro_score += 0.2   (close to or below target, good)
   - Else if cpi_yoy < 1.0 → macro_score -= 0.1         (too low inflation can mean weak demand)

   Growth (gdp_growth, if available):
   - If gdp_growth > 2.0 → macro_score += 0.2
   - If gdp_growth < 0.0 → macro_score -= 0.3

   Unemployment (if available):
   - If 3.5 <= unemployment <= 5.5 → macro_score += 0.1
   - If unemployment > 6.5 → macro_score -= 0.2

   After all adjustments, CLAMP macro_score to the range [-1, +1].

3) Define a macro REGIME from macro_score:
   - If macro_score >= 0.25 → regime = "good"
   - If -0.25 < macro_score < 0.25 → regime = "ok"
   - If macro_score <= -0.25 → regime = "bad"

4) Define macro_sentiment (Fed stance):
   - "hawkish": inflation clearly above target and/or communication focused on tightening,
                or policy rate high relative to inflation.
   - "dovish": inflation near target or falling AND communication suggests easing / rate cuts.
   - "neutral": anything in between.

5) In the explanation, speak in SIMPLE terms for a non-economist:
   - Say if inflation is “hot / under control / very low”.
   - Say if growth is “strong / moderate / weak”.
   - Say if labor market is “tight / normal / soft”.
   - Say what that usually means for the S&P 500 (supportive vs. risky backdrop).

Return ONLY valid JSON in this exact structure:

{{
  "macro_sentiment": "hawkish",
  "macro_score": 0.31,
  "regime": "good",
  "drivers": {{
    "cpi_mom": 0.3,
    "cpi_yoy": 2.9,
    "unemployment": 4.4,
    "gdp_growth": 3.8,
    "policy_rate_midpoint": 3.88
  }},
  "explanation": "Short explanation here.",
  "as_of": "{as_of_date}"
}}
"""

    resp = llm.invoke(prompt)
    content = resp.content.strip()

    try:
        data = json.loads(content)
    except Exception:
        # Fallback if the model didn't return clean JSON
        data = {
            "macro_sentiment": "neutral",
            "macro_score": 0.0,
            "regime": "ok",
            "drivers": {
                "cpi_mom": None,
                "cpi_yoy": None,
                "unemployment": None,
                "gdp_growth": None,
                "policy_rate_midpoint": None,
            },
            "explanation": content[:500],
            "as_of": str(as_of_date),
        }

    # Ensure as_of is there even if the model forgot
    if "as_of" not in data:
        data["as_of"] = str(as_of_date)

    return data

# analisys step by step

In [34]:
# out = Analyze_stock("spx stock price prediction for this week and next week")

from pprint import pprint
import json
import pandas as pd

pd.set_option("display.max_colwidth", 200)   # wrap long text instead of truncating
pd.set_option("display.width", 120)          # max width of table
pd.set_option("display.max_columns", None)   # show all columns

def pretty_json(data):
    print(json.dumps(data, indent=2, ensure_ascii=False))
# =========================================
# 1) WEEKLY ANALYSIS REPORT (BASED ON TODAY)
# =========================================

# 1. Get today's calendar date
today = pd.Timestamp.today().normalize()      # e.g. 2025-12-06 00:00:00

# 2. Ask analyze_week to analyze "this week" based on today's date
#    analyze_week will internally:
#    - clip to the last available date in df_weekly if data is behind
#    - pick the correct week index (Friday <= today)
report, hist = analyze_week(df_weekly, analysis_week_end=today)

# 3. The real week actually analyzed (could be last Friday if data is behind)
week_end_date = report["week_index"].date()

display("\n===== WEEKLY ANALYSIS REPORT (JSON) =====")
display(json.dumps(report, indent=2, default=str))

display("\n==============================")
display(" CURRENT WEEK ANALYSIS (BASED ON TODAY)")
display("==============================")
display(f"Today:           {today.date()}")
display(f"Week analyzed:   {week_end_date} (ISO week {report['requested_week_number']})")
display("==============================\n")



# =========================================
# 2) LLM NEWS SENTIMENT ANALYSIS
# =========================================
news_headlines = fetch_headlines()

sentiment = summarize_market_sentiment(news_headlines, week_end_date)

display("\n=== NEWS SENTIMENT (LLM BASED ON MARKET HEADLINES) ===")
display(json.dumps(sentiment, indent=2, default=str))


# =========================================
# 3) HISTORICAL MODEL (WEEKLY FORECAST)
# =========================================
probability_next_week_up = report["forecast"]["probability_next_week_up_all_history"] > 0.5


# =========================================
# 4) COMBINED INTERPRETATION
# =========================================
sentiment_label = sentiment.get("label", "neutral").lower()
sentiment_score = float(sentiment.get("score", 0.0))
explanation = sentiment.get("explanation", "")




display("\n=== COMBINED INTERPRETATION ===")

if direction_hist == "UP" and "bull" in sentiment_label:
    display(" Historical data and news sentiment align → **Bullish scenario**.")
elif direction_hist == "DOWN" and "bear" in sentiment_label:
    display("Historical data and news sentiment align → **Bearish scenario**.")
else:
    display(" Mixed signals → **Uncertain scenario**.")

display("\nLLM Explanation:")
display(explanation[:600])




url = "https://data.bls.gov/timeseries/CUSR0000SA0&output_view=pct_1mth"
page_text = fetch_fed_page(url)
macro = summarize_macro_fomc(page_text, as_of_date=pd.Timestamp.today())
pretty_json(macro)


'\n===== WEEKLY ANALYSIS REPORT (JSON) ====='

'{\n  "analysis_week_end": "2025-12-12 00:00:00",\n  "week_index": "2025-12-12 00:00:00",\n  "requested_date": "2025-12-07 00:00:00",\n  "requested_week_number": 49,\n  "data_first_available": "2000-01-28 00:00:00",\n  "data_last_available": "2025-11-21 00:00:00",\n  "gap_weeks": 3,\n  "years_back_param": 3,\n  "window_start": "2022-12-23 00:00:00",\n  "window_end": "2025-11-21 00:00:00",\n  "historical_stats": {\n    "num_samples": 2,\n    "mean_weekly_return": 0.005858205000900618,\n    "std_weekly_return": 0.005286809691068925,\n    "min_weekly_return": 0.002119866017503025,\n    "max_weekly_return": 0.009596543984298211\n  },\n  "forecast": {\n    "probability_next_week_up_all_history": 0.5,\n    "probability_next_week_up_last_year": 0.5306122448979592\n  }\n}'

'\n=============================='

' CURRENT WEEK ANALYSIS (BASED ON TODAY)'

'=============================='

...

' Mixed signals → **Uncertain scenario**.'

'\nLLM Explanation:'

'The headlines present mixed signals: some names (e.g., FCEL) show gains while others (SentinelOne, Parsons) are down, and the Netflix/Microsoft references without clear directional context don’t imply a broad market thrust. There’s no evident breadth or macro catalyst pointing to a sustained bullish or bearish trend for the S&P 500 in the week.'

d:\Python310\lib\site-packages\bs4\builder\_lxml.py:124: DeprecationWarning: The 'strip_cdata' option of HTMLParser() has never done anything and will eventually be removed.
  parser = parser(


{
  "macro_sentiment": "hawkish",
  "score": 0.64,
  "key_events": [
    "2025 Jan CPI 1-month change +0.5% indicates persistent inflation pressure in the near term.",
    "2025 Mar CPI monthly change -0.1% shows occasional soft readings but not a sustained downtrend.",
    "2025 Aug-Sep monthly changes around +0.3% to +0.4% keep inflation sticky rather than clearly cooling.",
    "Data cover All items CPI (not core) and are current as of Dec 7, 2025, reinforcing that broad inflation remains a policy-relevant signal."
  ],
  "explanation": "Persistent, broad inflation pressures imply the Fed would be cautious about rate cuts and may keep policy stance restrictive for longer. This tends to weigh on the S&P 500 due to higher discount rates and potential slower growth, unless inflation decisively slows and expectations shift."
}
